# Evaluation Notebook — using `evaluation_ab.py`

This notebook runs your **exact** `evaluation_ab.py` evaluation pipeline in Colab.

It is set up for the **own-patch** evaluation supported by that script:

- `mAP50`, `mAP50:95`, `Precision`, `Recall` via `yolo val`
- detection density metrics via `model.predict`
- false-positive metrics via GT matching

## Notes
- This notebook uses your script **as-is**.
- Because the script loads the model with `YOLO(args.model)`, it is best suited to **YOLO-family checkpoints**.
- If you want **RT-DETR** with the same workflow, the script would need a small update.

In [ ]:
# CELL 1 — Mount Drive & install dependencies
from google.colab import drive
drive.mount('/content/drive')

!pip install -q ultralytics numpy pandas
import ultralytics, torch
print("Ultralytics:", ultralytics.__version__)
print("Torch:", torch.__version__)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only")

Normal evaluation

In [ ]:
%%writefile /content/evaluation_ab.py

import argparse, json, re, subprocess, sys
from pathlib import Path
import numpy as np
from ultralytics import YOLO

IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}


# ----------------- utils -----------------
def run(cmd, log_path: Path):
    log_path.parent.mkdir(parents=True, exist_ok=True)
    with log_path.open("w", encoding="utf-8") as f:
        p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
        for line in p.stdout:
            sys.stdout.write(line)
            f.write(line)
        rc = p.wait()
    return rc


def percentile(arr, p):
    if arr is None or len(arr) == 0:
        return None
    return float(np.percentile(np.asarray(arr, dtype=np.float32), p))


def mean_std(values):
    vals = [v for v in values if isinstance(v, (int, float)) and not np.isnan(v)]
    if not vals:
        return {"mean": None, "std": None, "n": 0}
    a = np.asarray(vals, dtype=np.float32)
    return {"mean": float(a.mean()), "std": float(a.std(ddof=1)) if len(a) > 1 else 0.0, "n": int(len(a))}


def safe_mkdir(p: Path):
    p.mkdir(parents=True, exist_ok=True)
    return p


def model_names_as_list(model: YOLO):
    """
    Ultralytics model.names is usually dict {id: name}. Convert to list indexed by id.
    """
    names = getattr(model, "names", None)
    if names is None:
        raise ValueError("Model has no .names; cannot build dataset YAML (needs names/nc).")

    if isinstance(names, dict):
        max_k = max(int(k) for k in names.keys())
        out = [None] * (max_k + 1)
        for k, v in names.items():
            out[int(k)] = str(v)
        for i, v in enumerate(out):
            if v is None:
                out[i] = f"class_{i}"
        return out

    if isinstance(names, (list, tuple)):
        return [str(x) for x in names]

    raise ValueError(f"Unsupported model.names type: {type(names)}")


def read_dataset_yaml(yaml_path: Path):
    txt = yaml_path.read_text(encoding="utf-8", errors="ignore").splitlines()
    d = {"path": None, "train": None, "val": None, "test": None}
    for ln in txt:
        ln = ln.strip()
        if not ln or ln.startswith("#"):
            continue
        for k in ("path", "train", "val", "test"):
            if ln.startswith(k + ":"):
                d[k] = ln.split(":", 1)[1].strip()
    return d


def _has_images(dir_path: Path) -> bool:
    if not dir_path.exists() or not dir_path.is_dir():
        return False
    for x in dir_path.iterdir():
        if x.is_file() and x.suffix.lower() in IMG_EXTS:
            return True
    return False


def get_image_dir_from_yaml(yaml_path: Path) -> Path:
    """
    Robust: supports:
      test: images/test  (preferred)
      test: images       + actual images in images/test
      test: images       + images directly in images/
    Falls back: test -> val -> train.
    """
    y = read_dataset_yaml(yaml_path)
    root = y.get("path")
    if not root:
        raise ValueError(f"Could not parse 'path:' from {yaml_path}")

    rel = y.get("test") or y.get("val") or y.get("train")
    if not rel:
        raise ValueError(f"Could not parse test/val/train from {yaml_path}")

    base = (Path(root) / rel).resolve()

    if _has_images(base):
        return base

    for sub in ("test", "val", "train"):
        cand = base / sub
        if _has_images(cand):
            return cand

    if base.name != "images":
        cand = (Path(root) / "images").resolve()
        if _has_images(cand):
            return cand
        for sub in ("test", "val", "train"):
            if _has_images(cand / sub):
                return cand / sub

    return base


def get_labels_dir_for_images(img_dir: Path) -> Path:
    parts = list(img_dir.parts)
    if "images" in parts:
        idx = parts.index("images")
        parts[idx] = "labels"
        return Path(*parts)
    return img_dir.parent / "labels"


def write_yaml(out_yaml: Path, root: Path, test_rel: str, names):
    out_yaml.parent.mkdir(parents=True, exist_ok=True)
    content = (
        f"path: {root}\n"
        f"train: {test_rel}\n"
        f"val: {test_rel}\n"
        f"test: {test_rel}\n"
        f"\nnc: {len(names)}\n"
        f"names: {json.dumps(names)}\n"
    )
    out_yaml.write_text(content, encoding="utf-8")


def parse_yolo_val_metrics_from_log(log_path: Path):
    pat_all = re.compile(r"^\s*all\s+(\d+)\s+(\d+)\s+(\d*\.?\d+)\s+(\d*\.?\d+)\s+(\d*\.?\d+)\s+(\d*\.?\d+)\s*$")
    per_class = {}
    pat_cls = re.compile(r"^\s*([A-Za-z0-9 _-]+)\s+(\d+)\s+(\d+)\s+(\d*\.?\d+)\s+(\d*\.?\d+)\s+(\d*\.?\d+)\s+(\d*\.?\d+)\s*$")

    images = instances = None
    P = R = mAP50 = mAP5095 = None

    for ln in log_path.read_text(encoding="utf-8", errors="ignore").splitlines():
        m = pat_all.match(ln)
        if m:
            images = int(m.group(1))
            instances = int(m.group(2))
            P = float(m.group(3))
            R = float(m.group(4))
            mAP50 = float(m.group(5))
            mAP5095 = float(m.group(6))

        mc = pat_cls.match(ln)
        if mc:
            cls = mc.group(1).strip()
            if cls != "all":
                per_class[cls] = {
                    "images": int(mc.group(2)),
                    "instances": int(mc.group(3)),
                    "precision": float(mc.group(4)),
                    "recall": float(mc.group(5)),
                    "mAP50": float(mc.group(6)),
                    "mAP50-95": float(mc.group(7)),
                }

    return {
        "images": images,
        "instances": instances,
        "precision": P,
        "recall": R,
        "mAP50": mAP50,
        "mAP50-95": mAP5095,
        "per_class": per_class,
        "log_path": str(log_path),
    }


# ----------------- A) detection density / hallucination-ish -----------------
def compute_detection_density_metrics(model: YOLO, yaml_path: Path, imgsz: int, conf_min: float, iou: float, thresholds, det_k_list):
    img_dir = get_image_dir_from_yaml(yaml_path)
    names = model_names_as_list(model)

    det_counts = []
    max_conf_any = []
    has_ge_k = {str(k): 0 for k in det_k_list}
    has_any_conf_ge = {str(t): 0 for t in thresholds}
    total_dets_conf_ge = {str(t): 0 for t in thresholds}
    class_hist = {}

    results_iter = model.predict(
        source=str(img_dir),
        imgsz=imgsz,
        conf=conf_min,
        iou=iou,
        device="cpu",
        stream=True,
        verbose=False,
    )

    n_images = 0
    for r in results_iter:
        n_images += 1
        boxes = r.boxes
        n = 0 if boxes is None else len(boxes)
        det_counts.append(n)

        if n == 0:
            max_conf_any.append(0.0)
        else:
            confs = boxes.conf.detach().cpu().numpy().astype(np.float32)
            max_conf_any.append(float(confs.max()))
            clss = boxes.cls.detach().cpu().numpy().astype(int)

            for c in clss:
                k = names[c] if (0 <= c < len(names)) else str(int(c))
                class_hist[k] = class_hist.get(k, 0) + 1

            for t in thresholds:
                total_dets_conf_ge[str(t)] += int((confs >= t).sum())

        for k in det_k_list:
            if n >= k:
                has_ge_k[str(k)] += 1

        for t in thresholds:
            if max_conf_any[-1] >= t:
                has_any_conf_ge[str(t)] += 1

        if n_images == 1 or (n_images % 10 == 0):
            print(f"[A] {yaml_path.name}: processed {n_images} images", flush=True)

    det_counts_np = np.asarray(det_counts, dtype=np.float32)
    max_conf_np = np.asarray(max_conf_any, dtype=np.float32)

    return {
        "images_eval": int(n_images),
        "detections_total_model": int(det_counts_np.sum()),
        "detections_per_image_mean": float(det_counts_np.mean()) if n_images else None,
        "detections_per_image_median": float(np.median(det_counts_np)) if n_images else None,
        "detections_per_image_p95": percentile(det_counts_np, 95),
        "max_conf_any_mean": float(max_conf_np.mean()) if n_images else None,
        "max_conf_any_median": float(np.median(max_conf_np)) if n_images else None,
        "max_conf_any_p95": percentile(max_conf_np, 95),
        "image_rate_dets_ge": {k: (v / n_images if n_images else None) for k, v in has_ge_k.items()},
        "image_rate_any_det_conf_ge": {k: (v / n_images if n_images else None) for k, v in has_any_conf_ge.items()},
        "total_detections_conf_ge": total_dets_conf_ge,
        "predicted_class_histogram": dict(sorted(class_hist.items(), key=lambda kv: kv[1], reverse=True)),
        "config": {
            "yaml": str(yaml_path),
            "img_dir_used": str(get_image_dir_from_yaml(yaml_path)),
            "imgsz": imgsz,
            "conf_min": conf_min,
            "iou": iou,
            "thresholds": thresholds,
            "det_k_list": det_k_list,
        },
    }


# ----------------- B) false positives vs GT -----------------
def read_yolo_gt_labels(txt_path: Path):
    if not txt_path.exists():
        return []
    lines = txt_path.read_text(encoding="utf-8", errors="ignore").strip().splitlines()
    out = []
    for ln in lines:
        parts = ln.strip().split()
        if len(parts) < 5:
            continue
        try:
            c = int(float(parts[0]))
            x, y, w, h = map(float, parts[1:5])
            out.append((c, x, y, w, h))
        except Exception:
            continue
    return out


def xywhn_to_xyxy(x, y, w, h):
    x1 = x - w / 2.0
    y1 = y - h / 2.0
    x2 = x + w / 2.0
    y2 = y + h / 2.0
    return x1, y1, x2, y2


def iou_xyxy(a, b):
    ax1, ay1, ax2, ay2 = a
    bx1, by1, bx2, by2 = b
    ix1 = max(ax1, bx1)
    iy1 = max(ay1, by1)
    ix2 = min(ax2, bx2)
    iy2 = min(ay2, by2)
    iw = max(0.0, ix2 - ix1)
    ih = max(0.0, iy2 - iy1)
    inter = iw * ih
    area_a = max(0.0, ax2 - ax1) * max(0.0, ay2 - ay1)
    area_b = max(0.0, bx2 - bx1) * max(0.0, by2 - by1)
    union = area_a + area_b - inter
    return inter / union if union > 0 else 0.0


def compute_false_positive_metrics(model: YOLO, yaml_path: Path, imgsz: int, conf_min: float, iou: float, thresholds, gt_iou_match: float):
    img_dir = get_image_dir_from_yaml(yaml_path)
    lab_dir = get_labels_dir_for_images(img_dir)

    fp_per_image = []
    fp_by_thr = {str(t): [] for t in thresholds}

    total_fp = 0
    total_tp = 0
    total_preds = 0
    total_gt = 0

    results_iter = model.predict(
        source=str(img_dir),
        imgsz=imgsz,
        conf=conf_min,
        iou=iou,
        device="cpu",
        stream=True,
        verbose=False,
    )

    n_images = 0
    for r in results_iter:
        n_images += 1
        img_name = Path(r.path).name
        stem = Path(img_name).stem
        gt_path = lab_dir / f"{stem}.txt"

        gt = read_yolo_gt_labels(gt_path)
        total_gt += len(gt)

        boxes = r.boxes
        if boxes is None or len(boxes) == 0:
            fp_per_image.append(0)
            for t in thresholds:
                fp_by_thr[str(t)].append(0)
            if n_images == 1 or (n_images % 10 == 0):
                print(f"[B] {yaml_path.name}: processed {n_images} images", flush=True)
            continue

        pred_cls = boxes.cls.detach().cpu().numpy().astype(int)
        pred_xywhn = boxes.xywhn.detach().cpu().numpy().astype(np.float32)
        pred_conf = boxes.conf.detach().cpu().numpy().astype(np.float32)

        preds = []
        for i in range(len(pred_cls)):
            c = int(pred_cls[i])
            x, y, w, h = pred_xywhn[i].tolist()
            preds.append((c, xywhn_to_xyxy(x, y, w, h), float(pred_conf[i])))

        total_preds += len(preds)

        gt_by_class = {}
        for (c, x, y, w, h) in gt:
            gt_by_class.setdefault(int(c), []).append(xywhn_to_xyxy(x, y, w, h))

        def count_fp_at(thr):
            pf = [(c, box, conf) for (c, box, conf) in preds if conf >= thr]
            pf.sort(key=lambda z: z[2], reverse=True)

            tp = 0
            fp = 0
            remaining = {c: list(lst) for c, lst in gt_by_class.items()}

            for c, pbox, _ in pf:
                best_iou = 0.0
                best_j = None
                gtl = remaining.get(c, [])
                for j, gbox in enumerate(gtl):
                    iouv = iou_xyxy(pbox, gbox)
                    if iouv > best_iou:
                        best_iou = iouv
                        best_j = j
                if best_j is not None and best_iou >= gt_iou_match:
                    tp += 1
                    gtl.pop(best_j)
                    remaining[c] = gtl
                else:
                    fp += 1
            return tp, fp, len(pf)

        tp0, fp0, _ = count_fp_at(conf_min)
        total_tp += tp0
        total_fp += fp0
        fp_per_image.append(fp0)

        for t in thresholds:
            _, fpt, _ = count_fp_at(t)
            fp_by_thr[str(t)].append(fpt)

        if n_images == 1 or (n_images % 10 == 0):
            print(f"[B] {yaml_path.name}: processed {n_images} images", flush=True)

    fp_arr = np.asarray(fp_per_image, dtype=np.float32)

    return {
        "images_eval": int(n_images),
        "gt_total_boxes": int(total_gt),
        "pred_total_boxes_conf_ge_conf_min": int(total_preds),
        "tp_total_conf_ge_conf_min": int(total_tp),
        "fp_total_conf_ge_conf_min": int(total_fp),
        "fp_per_image_mean_conf_min": float(fp_arr.mean()) if n_images else None,
        "fp_per_image_median_conf_min": float(np.median(fp_arr)) if n_images else None,
        "fp_per_image_p95_conf_min": percentile(fp_arr, 95),
        "fp_per_image_conf_ge": {str(t): float(np.mean(fp_by_thr[str(t)])) if n_images else None for t in thresholds},
        "config": {
            "yaml": str(yaml_path),
            "img_dir_used": str(img_dir),
            "labels_dir_used": str(lab_dir),
            "imgsz": imgsz,
            "conf_min": conf_min,
            "iou": iou,
            "thresholds": thresholds,
            "gt_iou_match": gt_iou_match,
        },
    }


def delta_numeric(a, b):
    if isinstance(a, (int, float)) and isinstance(b, (int, float)):
        return b - a
    return None


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--out_root", default="/home/nvidia/nova_edge_eval/yolo_eval_runs")
    ap.add_argument("--run_name", default=None)
    ap.add_argument("--model", required=True)

    ap.add_argument("--clean_root", required=True)
    ap.add_argument("--patched_parent", required=True)

    ap.add_argument("--imgsz", type=int, default=672)
    ap.add_argument("--conf_min", type=float, default=0.001)
    ap.add_argument("--iou", type=float, default=0.6)
    ap.add_argument("--thresholds", default="0.3,0.5,0.7")
    ap.add_argument("--gt_iou_match", type=float, default=0.5)

    ap.add_argument("--scenarios", default="1,2,3,4,5,6,7,8,9")
    ap.add_argument("--det_k", default="1,5,10,20,50")
    args = ap.parse_args()

    thresholds = [float(x.strip()) for x in args.thresholds.split(",") if x.strip()]
    det_k_list = [int(x.strip()) for x in args.det_k.split(",") if x.strip()]

    out_root = Path(args.out_root)
    out_root.mkdir(parents=True, exist_ok=True)
    if args.run_name:
        run_dir = out_root / args.run_name
    else:
        from datetime import datetime
        ts = datetime.now().strftime("%Y%m%d_%H%M%S")
        run_dir = out_root / f"run_{ts}"
    safe_mkdir(run_dir)

    yamls_dir = safe_mkdir(run_dir / "yamls")

    print(f"RUN DIR: {run_dir}", flush=True)
    print(f"MODEL: {args.model}", flush=True)
    print(f"IMG_SIZE: {args.imgsz}", flush=True)

    model = YOLO(args.model)
    names = model_names_as_list(model)  # ✅ FIX: always provide names/nc in YAMLs

    scenarios = [f"billboard{int(s):02d}" for s in args.scenarios.split(",") if s.strip()]

    per_scenario = []
    clean_map50s = []; patched_map50s = []; delta_map50s = []
    clean_map5095s = []; patched_map5095s = []; delta_map5095s = []
    clean_det_means = []; patched_det_means = []; delta_det_means = []
    clean_fp_means = []; patched_fp_means = []; delta_fp_means = []

    for bb in scenarios:
        sc_dir = safe_mkdir(run_dir / bb)

        clean_img_dir = Path(args.clean_root) / "images" / bb
        clean_lab_dir = Path(args.clean_root) / "labels" / bb
        if not clean_img_dir.exists():
            raise FileNotFoundError(f"[{bb}] Missing clean images: {clean_img_dir}")
        if not clean_lab_dir.exists():
            raise FileNotFoundError(f"[{bb}] Missing clean labels: {clean_lab_dir}")

        clean_yaml = yamls_dir / f"{bb}_clean.yaml"
        write_yaml(clean_yaml, Path(args.clean_root), f"images/{bb}", names)

        patched_root = Path(args.patched_parent) / f"patched_dataset{int(bb[-2:])}"
        if not patched_root.exists():
            raise FileNotFoundError(f"[{bb}] Missing patched root: {patched_root}")

        patched_yaml = yamls_dir / f"{bb}_patched.yaml"
        write_yaml(patched_yaml, patched_root, "images", names)

        clean_val_log = sc_dir / f"{bb}_clean_val.log"
        patched_val_log = sc_dir / f"{bb}_patched_val.log"

        cmd_base = [
            "yolo", "val",
            f"model={args.model}",
            f"imgsz={args.imgsz}",
            f"conf={args.conf_min}",
            f"iou={args.iou}",
            "split=test",
            "batch=1",
            "workers=0",
            f"project={sc_dir}",
        ]

        rc = run(cmd_base + [f"data={clean_yaml}", f"name={bb}_clean"], clean_val_log)
        if rc != 0:
            raise SystemExit(f"yolo val clean failed for {bb} (rc={rc}) see {clean_val_log}")

        rc = run(cmd_base + [f"data={patched_yaml}", f"name={bb}_patched"], patched_val_log)
        if rc != 0:
            raise SystemExit(f"yolo val patched failed for {bb} (rc={rc}) see {patched_val_log}")

        clean_val = parse_yolo_val_metrics_from_log(clean_val_log)
        patched_val = parse_yolo_val_metrics_from_log(patched_val_log)

        clean_A = compute_detection_density_metrics(model, clean_yaml, args.imgsz, args.conf_min, args.iou, thresholds, det_k_list)
        patched_A = compute_detection_density_metrics(model, patched_yaml, args.imgsz, args.conf_min, args.iou, thresholds, det_k_list)

        clean_B = compute_false_positive_metrics(model, clean_yaml, args.imgsz, args.conf_min, args.iou, thresholds, args.gt_iou_match)
        patched_B = compute_false_positive_metrics(model, patched_yaml, args.imgsz, args.conf_min, args.iou, thresholds, args.gt_iou_match)

        delta = {
            "yolo_val": {
                "precision": delta_numeric(clean_val.get("precision"), patched_val.get("precision")),
                "recall": delta_numeric(clean_val.get("recall"), patched_val.get("recall")),
                "mAP50": delta_numeric(clean_val.get("mAP50"), patched_val.get("mAP50")),
                "mAP50-95": delta_numeric(clean_val.get("mAP50-95"), patched_val.get("mAP50-95")),
            },
            "A": {
                "detections_per_image_mean": delta_numeric(clean_A.get("detections_per_image_mean"), patched_A.get("detections_per_image_mean")),
                "detections_total_model": delta_numeric(clean_A.get("detections_total_model"), patched_A.get("detections_total_model")),
            },
            "B": {
                "fp_per_image_mean_conf_min": delta_numeric(clean_B.get("fp_per_image_mean_conf_min"), patched_B.get("fp_per_image_mean_conf_min")),
                "fp_total_conf_ge_conf_min": delta_numeric(clean_B.get("fp_total_conf_ge_conf_min"), patched_B.get("fp_total_conf_ge_conf_min")),
            },
        }

        entry = {
            "scenario": bb,
            "patched_root": str(patched_root),
            "model": args.model,
            "imgsz": args.imgsz,
            "clean": {"yaml": str(clean_yaml), "yolo_val": clean_val, "A": clean_A, "B": clean_B},
            "patched": {"yaml": str(patched_yaml), "yolo_val": patched_val, "A": patched_A, "B": patched_B},
            "delta_patched_minus_clean": delta,
        }
        per_scenario.append(entry)
        (sc_dir / "metrics.json").write_text(json.dumps(entry, indent=2), encoding="utf-8")

        cm50 = clean_val.get("mAP50"); pm50 = patched_val.get("mAP50")
        cm95 = clean_val.get("mAP50-95"); pm95 = patched_val.get("mAP50-95")
        clean_map50s.append(cm50); patched_map50s.append(pm50); delta_map50s.append(delta_numeric(cm50, pm50))
        clean_map5095s.append(cm95); patched_map5095s.append(pm95); delta_map5095s.append(delta_numeric(cm95, pm95))

        cdet = clean_A.get("detections_per_image_mean"); pdet = patched_A.get("detections_per_image_mean")
        clean_det_means.append(cdet); patched_det_means.append(pdet); delta_det_means.append(delta_numeric(cdet, pdet))

        cfp = clean_B.get("fp_per_image_mean_conf_min"); pfp = patched_B.get("fp_per_image_mean_conf_min")
        clean_fp_means.append(cfp); patched_fp_means.append(pfp); delta_fp_means.append(delta_numeric(cfp, pfp))

    summary = {
        "run_dir": str(run_dir),
        "model": args.model,
        "imgsz": args.imgsz,
        "scenarios": scenarios,
        "mean_std": {
            "clean_mAP50": mean_std(clean_map50s),
            "patched_mAP50": mean_std(patched_map50s),
            "delta_mAP50": mean_std(delta_map50s),
            "clean_mAP50_95": mean_std(clean_map5095s),
            "patched_mAP50_95": mean_std(patched_map5095s),
            "delta_mAP50_95": mean_std(delta_map5095s),
            "clean_det_mean": mean_std(clean_det_means),
            "patched_det_mean": mean_std(patched_det_means),
            "delta_det_mean": mean_std(delta_det_means),
            "clean_fp_mean": mean_std(clean_fp_means),
            "patched_fp_mean": mean_std(patched_fp_means),
            "delta_fp_mean": mean_std(delta_fp_means),
        },
    }

    combined = {**summary, "per_scenario": per_scenario}

    out_combined = run_dir / "combined_metrics.json"
    out_summary = run_dir / "summary.json"
    out_combined.write_text(json.dumps(combined, indent=2), encoding="utf-8")
    out_summary.write_text(json.dumps(summary, indent=2), encoding="utf-8")

    print("\nWROTE:", out_combined, flush=True)
    print("WROTE:", out_summary, flush=True)
    print("\nSUMMARY:\n", json.dumps(summary, indent=2), flush=True)


if __name__ == "__main__":
    main()

In [ ]:
# CELL 3 — Paths and model checkpoints
from pathlib import Path
import os, json, pandas as pd

CLEAN_ROOT = "/content/drive/MyDrive/Patched Datasets/Clean"
PATCHED_PARENT = "/content/drive/MyDrive/Patched Datasets"
OUT_ROOT = "/content/drive/MyDrive/unified_eval_ab"

# YOLO-family checkpoints supported directly by evaluation_ab.py
MODEL_PATHS = {
    "yolov8n": "/content/drive/MyDrive/yolo_trained_models/2dod_checkpoints/best.pt",
    "yolov8s": "/content/drive/MyDrive/yolov8s_carlagear/train2/weights/best.pt",
    "yolov5n": "/content/drive/MyDrive/yolov5n_carlagear/train/weights/best.pt",
}

# Evaluation config
IMGSZ = 672
CONF_MIN = 0.001
IOU = 0.6
THRESHOLDS = "0.3,0.5,0.7"
GT_IOU_MATCH = 0.5
SCENARIOS = "1,2,3,4,5,6,7,8,9"
DET_K = "1,5,10,20,50"

Path(OUT_ROOT).mkdir(parents=True, exist_ok=True)

print("CLEAN_ROOT:", CLEAN_ROOT)
print("PATCHED_PARENT:", PATCHED_PARENT)
print("OUT_ROOT:", OUT_ROOT)
print("Models:", list(MODEL_PATHS.keys()))

In [ ]:
# CELL 3 — Paths and model checkpoints
from pathlib import Path
import os, json, pandas as pd

# ------------------------------------------------------------
# CROSS-SCENE TRANSFERABILITY SETUP
# Patch trained on Billboard 1, evaluated on other scenes
# Example folders:
#   /content/drive/MyDrive/Patch1BillX/Patch1Bill2
#   /content/drive/MyDrive/Patch1BillX/Patch1Bill3
#   ...
#   /content/drive/MyDrive/Patch1BillX/Patch1Bill9
# ------------------------------------------------------------

SOURCE_PATCH_SCENE = 1
TARGET_SCENARIOS = [2, 3, 4, 5, 6, 7, 8, 9]

# Clean dataset root for the corresponding clean scenes
# Change this only if your clean data is somewhere else
CLEAN_ROOT = "/content/drive/MyDrive/Patched Datasets/Clean"

# Root containing the transferred patched datasets
PATCHED_PARENT = "/content/drive/MyDrive/Patch1BillX"

# Output folder for this cross-scene experiment
OUT_ROOT = f"/content/drive/MyDrive/unified_eval_ab/cross_scene_patch{SOURCE_PATCH_SCENE}_to_{'-'.join(map(str, TARGET_SCENARIOS))}"

# YOLO-family checkpoints supported directly by evaluation_ab.py
MODEL_PATHS = {
    "yolov8n": "/content/drive/MyDrive/yolo_trained_models/2dod_checkpoints/best.pt",
    "yolov8s": "/content/drive/MyDrive/yolov8s_carlagear/train2/weights/best.pt",
    "yolov5n": "/content/drive/MyDrive/yolov5n_carlagear/train/weights/best.pt",
}

# Evaluation config
IMGSZ = 672
CONF_MIN = 0.001
IOU = 0.6
THRESHOLDS = "0.3,0.5,0.7"
GT_IOU_MATCH = 0.5

# Evaluate only transfer targets, not the source scene
SCENARIOS = ",".join(map(str, TARGET_SCENARIOS))

DET_K = "1,5,10,20,50"

Path(OUT_ROOT).mkdir(parents=True, exist_ok=True)

print("=== Cross-scene transferability evaluation ===")
print("Source patch scene:", SOURCE_PATCH_SCENE)
print("Target scenarios:", TARGET_SCENARIOS)
print("CLEAN_ROOT:", CLEAN_ROOT)
print("PATCHED_PARENT:", PATCHED_PARENT)
print("OUT_ROOT:", OUT_ROOT)
print("Models:", list(MODEL_PATHS.keys()))

In [ ]:
# CELL 4 — Sanity check dataset layout
from pathlib import Path

clean_root = Path(CLEAN_ROOT)
patched_parent = Path(PATCHED_PARENT)

print("Clean images exists:", (clean_root / "images").exists())
print("Clean labels exists:", (clean_root / "labels").exists())
print()

for i in range(1, 10):
    bb = f"billboard{i:02d}"
    clean_img = clean_root / "images" / bb
    clean_lab = clean_root / "labels" / bb
    patched = patched_parent / f"patched_dataset{i}"
    print(bb)
    print("  clean images:", clean_img.exists(), clean_img)
    print("  clean labels:", clean_lab.exists(), clean_lab)
    print("  patched root:", patched.exists(), patched)
    print()

## Run options

Use the next cell to test one model first.  
After that, run the full multi-model cell.

In [ ]:
# CELL 5 — Optional: debug one model / one scenario first
MODEL_KEY = "yolov8n"
DEBUG_RUN_NAME = "debug_yolov8n_bb01"

!python /content/evaluation_ab.py   --out_root "{OUT_ROOT}"   --run_name "{DEBUG_RUN_NAME}"   --model "{MODEL_PATHS[MODEL_KEY]}"   --clean_root "{CLEAN_ROOT}"   --patched_parent "{PATCHED_PARENT}"   --imgsz {IMGSZ}   --conf_min {CONF_MIN}   --iou {IOU}   --thresholds "{THRESHOLDS}"   --gt_iou_match {GT_IOU_MATCH}   --scenarios "1"   --det_k "{DET_K}"

In [ ]:
# CELL — Paths and config for cross-scene transferability
from pathlib import Path
import os, json, pandas as pd

# Patch source scene: patch generated on billboard01
PATCH_SOURCE = 1

# Evaluate transfer to these target scenes
TARGET_SCENARIOS = [2, 3, 4, 5, 6, 7, 8, 9]

# Clean dataset
CLEAN_ROOT = "/content/drive/MyDrive/Patched Datasets/Clean"

# Patched transfer dataset root
# Contains folders like:
#   Patch1Bill2, Patch1Bill3, ..., Patch1Bill9
PATCHED_PARENT = "/content/drive/MyDrive/Patch1BillX"

# Output root
OUT_ROOT = f"/content/drive/MyDrive/unified_eval_ab/cross_scene_patch{PATCH_SOURCE}_to_" + "-".join(map(str, TARGET_SCENARIOS))

# YOLO-family checkpoints supported directly by evaluation_ab.py
MODEL_PATHS = {
    "yolov8n": "/content/drive/MyDrive/yolo_trained_models/2dod_checkpoints/best.pt",
    "yolov8s": "/content/drive/MyDrive/yolov8s_carlagear/train2/weights/best.pt",
    "yolov5n": "/content/drive/MyDrive/yolov5n_carlagear/train/weights/best.pt",
}

# Evaluation config
IMGSZ = 672
CONF_MIN = 0.001
IOU = 0.6
THRESHOLDS = "0.3,0.5,0.7"
GT_IOU_MATCH = 0.5
SCENARIOS = ",".join(map(str, TARGET_SCENARIOS))
DET_K = "1,5,10,20,50"

Path(OUT_ROOT).mkdir(parents=True, exist_ok=True)

print("PATCH_SOURCE:", PATCH_SOURCE)
print("TARGET_SCENARIOS:", TARGET_SCENARIOS)
print("CLEAN_ROOT:", CLEAN_ROOT)
print("PATCHED_PARENT:", PATCHED_PARENT)
print("OUT_ROOT:", OUT_ROOT)
print("Models:", list(MODEL_PATHS.keys()))

In [ ]:
# CELL — Run cross-scene transferability evaluation
import subprocess
import sys

MODEL_KEY = "yolov5n"   # change to "yolov8s" or "yolov5n" if needed
RUN_NAME = f"cross_scene_{MODEL_KEY}_patch{PATCH_SOURCE}"

cmd = [
    sys.executable, "/content/evaluation_ab.py",
    "--out_root", OUT_ROOT,
    "--run_name", RUN_NAME,
    "--model", MODEL_PATHS[MODEL_KEY],
    "--clean_root", CLEAN_ROOT,
    "--patched_parent", PATCHED_PARENT,
    "--patch_source", str(PATCH_SOURCE),
    "--imgsz", str(IMGSZ),
    "--conf_min", str(CONF_MIN),
    "--iou", str(IOU),
    "--thresholds", THRESHOLDS,
    "--gt_iou_match", str(GT_IOU_MATCH),
    "--scenarios", SCENARIOS,
    "--det_k", DET_K,
]

print("Running command:\n", " ".join(cmd))
subprocess.run(cmd, check=True)

In [ ]:
MODEL_PATHS = {
    "yolov8n": "/content/drive/MyDrive/models/yolov8n_best.pt",  # source model
    "yolov8s": "/content/drive/MyDrive/models/yolov8s_best.pt",  # target model
    "yolov5n": "/content/drive/MyDrive/models/yolov5n_best.pt",  # target model
}

In [ ]:
CLEAN_ROOT = "/content/drive/MyDrive/Patched Datasets/Clean"
PATCHED_PARENT = "/content/drive/MyDrive/Patched Datasets"
OUT_ROOT = "/content/eval_runs_transfer"

In [ ]:
from pathlib import Path

CLEAN_ROOT = Path('/content/drive/MyDrive/Patched Datasets/Clean')
PATCH1_ROOT = Path('/content/drive/MyDrive/Patch1BillX')

# temporary folder shaped exactly how evaluation_ab.py expects it
TRANSFER_PATCHED_PARENT = Path('/content/transfer_patch1_parent')

MODEL_PATH = '/content/drive/MyDrive/YOUR_MODEL_FOLDER/best.pt'  # change this
OUT_ROOT = '/content/eval_runs_transfer'
RUN_NAME = 'patch1_cross_scenario_transfer'

In [ ]:
for i in range(1, 10):
    src = PATCH1_ROOT / f'Patch1Bill{i}'
    print(f'Patch1Bill{i}:', src.exists(), src)
    print('  images exists:', (src / 'images').exists())
    print('  labels exists:', (src / 'labels').exists())
    print('  images/test exists:', (src / 'images' / 'test').exists())
    print('  labels/test exists:', (src / 'labels' / 'test').exists())
    print()

In [ ]:
import os
import shutil

if TRANSFER_PATCHED_PARENT.exists():
    shutil.rmtree(TRANSFER_PATCHED_PARENT)

TRANSFER_PATCHED_PARENT.mkdir(parents=True, exist_ok=True)

for i in range(2, 10):
    src = PATCH1_ROOT / f'Patch1Bill{i}'
    dst = TRANSFER_PATCHED_PARENT / f'patched_dataset{i}'
    dst.mkdir(parents=True, exist_ok=True)

    # link the whole images and labels folders
    os.symlink(src / 'images', dst / 'images')
    os.symlink(src / 'labels', dst / 'labels')

print("Created:", TRANSFER_PATCHED_PARENT)
for p in sorted(TRANSFER_PATCHED_PARENT.iterdir()):
    print(p)

In [ ]:
for i in range(2, 10):
    p = TRANSFER_PATCHED_PARENT / f'patched_dataset{i}'
    print(f'patched_dataset{i}')
    print('  images:', (p / 'images').exists(), '->', p / 'images')
    print('  labels:', (p / 'labels').exists(), '->', p / 'labels')
    print('  images/test:', (p / 'images' / 'test').exists())
    print('  labels/test:', (p / 'labels' / 'test').exists())
    print()

In [ ]:
MODEL_PATHS = {
    'yolov8n': '/content/drive/MyDrive/yolo_trained_models/2dod_checkpoints/best.pt',
    'yolov8s': '/content/drive/MyDrive/yolov8s_carlagear/train2/weights/best.pt',
    'yolov5n': '/content/drive/MyDrive/yolov5n_carlagear/train/weights/best.pt',
}

TARGET_MODEL = 'yolov8n'   # change to 'yolov8s' or 'yolov5n' when needed
MODEL_PATH = MODEL_PATHS[TARGET_MODEL]

CLEAN_ROOT = '/content/drive/MyDrive/Patched Datasets/Clean'
PATCH1_ROOT = '/content/drive/MyDrive/Patch1BillX'
OUT_ROOT = '/content/eval_runs_transfer'
RUN_NAME = f'patch1_cross_scenario_transfer_{TARGET_MODEL}'

print("Using model:", TARGET_MODEL)
print("Model path:", MODEL_PATH)

In [ ]:
from pathlib import Path
import os, shutil, json, pandas as pd

MODEL_PATHS = {
    'yolov8n': '/content/drive/MyDrive/yolo_trained_models/2dod_checkpoints/best.pt',
    'yolov8s': '/content/drive/MyDrive/yolov8s_carlagear/train2/weights/best.pt',
    'yolov5n': '/content/drive/MyDrive/yolov5n_carlagear/train/weights/best.pt',
}

CLEAN_ROOT = Path('/content/drive/MyDrive/Patched Datasets/Clean')
OWN_PATCH_PARENT = Path('/content/drive/MyDrive/Patched Datasets')   # has patched_dataset1..9
PATCH1_ROOT = Path('/content/drive/MyDrive/Patch1BillX')             # has Patch1Bill2..9
OUT_ROOT = '/content/eval_runs_transfer'
SCENARIOS = "1,2,3,4,5,6,7,8,9"   # use all, with billboard01 taken from original patched_dataset1

In [ ]:
from pathlib import Path
import os, shutil

for model_name in MODEL_PATHS.keys():
    transfer_parent = Path(f'/content/transfer_patch1_parent_{model_name}')
    if transfer_parent.exists():
        shutil.rmtree(transfer_parent)
    transfer_parent.mkdir(parents=True, exist_ok=True)

    for i in range(1, 10):
        dst = transfer_parent / f'patched_dataset{i}'
        dst.mkdir(parents=True, exist_ok=True)

        # billboard01: use the original own-patch dataset, since Patch1Bill1 is missing
        if i == 1:
            src = OWN_PATCH_PARENT / 'patched_dataset1'
        else:
            src = PATCH1_ROOT / f'Patch1Bill{i}'

        if not src.exists():
            raise FileNotFoundError(f'Missing source folder for scenario {i}: {src}')

        img_src = src / 'images'
        lab_src = src / 'labels'

        if not img_src.exists():
            raise FileNotFoundError(f'Missing images folder for scenario {i}: {img_src}')
        if not lab_src.exists():
            raise FileNotFoundError(f'Missing labels folder for scenario {i}: {lab_src}')

        os.symlink(img_src, dst / 'images')
        os.symlink(lab_src, dst / 'labels')

    print(f'Created merged transfer parent for {model_name}: {transfer_parent}')

In [ ]:
from pathlib import Path
import os, shutil

for model_name in MODEL_PATHS.keys():
    transfer_parent = Path(f'/content/transfer_patch1_parent_{model_name}')
    if transfer_parent.exists():
        shutil.rmtree(transfer_parent)
    transfer_parent.mkdir(parents=True, exist_ok=True)

    for i in range(1, 10):
        dst = transfer_parent / f'patched_dataset{i}'
        dst.mkdir(parents=True, exist_ok=True)

        # billboard01: use the original own-patch dataset, since Patch1Bill1 is missing
        if i == 1:
            src = OWN_PATCH_PARENT / 'patched_dataset1'
        else:
            src = PATCH1_ROOT / f'Patch1Bill{i}'

        if not src.exists():
            raise FileNotFoundError(f'Missing source folder for scenario {i}: {src}')

        img_src = src / 'images'
        lab_src = src / 'labels'

        if not img_src.exists():
            raise FileNotFoundError(f'Missing images folder for scenario {i}: {img_src}')
        if not lab_src.exists():
            raise FileNotFoundError(f'Missing labels folder for scenario {i}: {lab_src}')

        os.symlink(img_src, dst / 'images')
        os.symlink(lab_src, dst / 'labels')

    print(f'Created merged transfer parent for {model_name}: {transfer_parent}')

In [ ]:
!find "/content/drive/MyDrive/Patched Datasets" -name "*.cache" -delete
!find "/content/drive/MyDrive/Patch1BillX" -name "*.cache" -delete
!find "/content/eval_runs_transfer" -name "*.cache" -delete

In [ ]:
from pathlib import Path

def audit_label_ids(label_dir):
    ids = set()
    bad_files = []
    for f in Path(label_dir).rglob("*.txt"):
        try:
            for line in f.read_text().splitlines():
                parts = line.strip().split()
                if len(parts) >= 5:
                    cid = int(float(parts[0]))
                    ids.add(cid)
        except Exception:
            bad_files.append(str(f))
    return sorted(ids), bad_files

for i in range(1, 10):
    clean_dir = Path(f"/content/drive/MyDrive/Patched Datasets/Clean/labels/billboard{i:02d}")
    patch_dir = Path(f"/content/drive/MyDrive/Patch1BillX/Patch1Bill{i}/labels/test")

    print(f"\n=== billboard{i:02d} ===")
    if clean_dir.exists():
        clean_ids, clean_bad = audit_label_ids(clean_dir)
        print("clean ids :", clean_ids)
    else:
        print("clean ids : MISSING")

    if patch_dir.exists():
        patch_ids, patch_bad = audit_label_ids(patch_dir)
        print("patch ids :", patch_ids)
    else:
        print("patch ids : MISSING")

In [ ]:
!find "/content/drive/MyDrive/Patched Datasets/Clean/labels" -name "*.cache" -delete
!find "/content/drive/MyDrive/Patched Datasets/patched_dataset1/labels" -name "*.cache" -delete
!find "/content/drive/MyDrive/Patched Datasets/patched_dataset2/labels" -name "*.cache" -delete
!find "/content/drive/MyDrive/Patched Datasets/patched_dataset3/labels" -name "*.cache" -delete
!find "/content/drive/MyDrive/Patched Datasets/patched_dataset4/labels" -name "*.cache" -delete
!find "/content/drive/MyDrive/Patched Datasets/patched_dataset5/labels" -name "*.cache" -delete
!find "/content/drive/MyDrive/Patched Datasets/patched_dataset6/labels" -name "*.cache" -delete
!find "/content/drive/MyDrive/Patched Datasets/patched_dataset7/labels" -name "*.cache" -delete
!find "/content/drive/MyDrive/Patched Datasets/patched_dataset8/labels" -name "*.cache" -delete
!find "/content/drive/MyDrive/Patched Datasets/patched_dataset9/labels" -name "*.cache" -delete
!find "/content/drive/MyDrive/Patch1BillX" -path "*/labels/*.cache" -delete
!find "/content/drive/MyDrive/Patch1BillX" -path "*/labels/test.cache" -delete

In [ ]:
from ultralytics import YOLO

model = YOLO('/content/drive/MyDrive/yolo_trained_models/2dod_checkpoints/best.pt')
print("num classes:", len(model.names))
print(model.names)

In [ ]:
from pathlib import Path
import shutil

CLEAN_ROOT = Path("/content/drive/MyDrive/Patched Datasets/Clean")
PATCH1_ROOT = Path("/content/drive/MyDrive/Patch1BillX")

for i in range(2, 10):
    clean_src = CLEAN_ROOT / "labels" / f"billboard{i:02d}"
    patch_dst = PATCH1_ROOT / f"Patch1Bill{i}" / "labels" / "test"

    if not clean_src.exists():
        print("missing clean labels:", clean_src)
        continue

    if patch_dst.exists():
        shutil.rmtree(patch_dst)
    patch_dst.mkdir(parents=True, exist_ok=True)

    for txt in clean_src.glob("*.txt"):
        shutil.copy2(txt, patch_dst / txt.name)

    print(f"replaced labels for Patch1Bill{i} from {clean_src}")

In [ ]:
from pathlib import Path

def label_ids(label_dir):
    ids = set()
    for f in Path(label_dir).glob("*.txt"):
        for line in f.read_text().splitlines():
            parts = line.strip().split()
            if len(parts) >= 5:
                ids.add(int(float(parts[0])))
    return sorted(ids)

for i in range(2, 10):
    clean_dir = Path(f"/content/drive/MyDrive/Patched Datasets/Clean/labels/billboard{i:02d}")
    patch_dir = Path(f"/content/drive/MyDrive/Patch1BillX/Patch1Bill{i}/labels/test")
    print(f"billboard{i:02d}")
    print("  clean:", label_ids(clean_dir))
    print("  patch:", label_ids(patch_dir))

In [ ]:
from pathlib import Path

for model_name, model_path in MODEL_PATHS.items():
    print(f'\n===== Running {model_name} =====')
    print("Exists:", Path(model_path).exists(), model_path)

    transfer_parent = Path(f'/content/transfer_patch1_parent_{model_name}')
    run_name = f'patch1_cross_scenario_transfer_{model_name}'

    !python /content/evaluation_ab.py \
      --out_root "{OUT_ROOT}" \
      --run_name "{run_name}" \
      --model "{model_path}" \
      --clean_root "{CLEAN_ROOT}" \
      --patched_parent "{transfer_parent}" \
      --imgsz 672 \
      --conf_min 0.001 \
      --iou 0.6 \
      --thresholds "0.3,0.5,0.7" \
      --gt_iou_match 0.5 \
      --scenarios "{SCENARIOS}" \
      --det_k "1,5,10,20,50"

In [ ]:
# CELL 8 — Load per-scenario results for one model
import json
import pandas as pd
from pathlib import Path

MODEL_TO_VIEW = "yolov8n"

combined_path = Path(OUT_ROOT) / f"{MODEL_TO_VIEW}_evaluation_ab_final" / "combined_metrics.json"
data = json.loads(combined_path.read_text())

rows = []
for s in data["per_scenario"]:
    rows.append({
        "Scenario": s["scenario"],
        "mAP50_C": s["clean"]["yolo_val"]["mAP50"],
        "mAP50_P": s["patched"]["yolo_val"]["mAP50"],
        "Δ_mAP50": s["delta_patched_minus_clean"]["yolo_val"]["mAP50"],
        "mAP50-95_C": s["clean"]["yolo_val"]["mAP50-95"],
        "mAP50-95_P": s["patched"]["yolo_val"]["mAP50-95"],
        "Δ_mAP50-95": s["delta_patched_minus_clean"]["yolo_val"]["mAP50-95"],
        "Precision_C": s["clean"]["yolo_val"]["precision"],
        "Precision_P": s["patched"]["yolo_val"]["precision"],
        "Recall_C": s["clean"]["yolo_val"]["recall"],
        "Recall_P": s["patched"]["yolo_val"]["recall"],
        "Det/img@0.001_C": s["clean"]["A"]["detections_per_image_mean"],
        "Det/img@0.001_P": s["patched"]["A"]["detections_per_image_mean"],
        "Δ_Det/img": s["delta_patched_minus_clean"]["A"]["detections_per_image_mean"],
        "FP/img@0.001_C": s["clean"]["B"]["fp_per_image_mean_conf_min"],
        "FP/img@0.001_P": s["patched"]["B"]["fp_per_image_mean_conf_min"],
        "Δ_FP/img": s["delta_patched_minus_clean"]["B"]["fp_per_image_mean_conf_min"],
    })

scenario_df = pd.DataFrame(rows)
scenario_df

In [ ]:
# CELL 9 — Export CSV files
from pathlib import Path

exports_dir = Path(OUT_ROOT) / "exports"
exports_dir.mkdir(parents=True, exist_ok=True)

summary_csv = exports_dir / "summary_across_models.csv"
summary_df.to_csv(summary_csv, index=False)

scenario_csv = exports_dir / f"{MODEL_TO_VIEW}_per_scenario.csv"
scenario_df.to_csv(scenario_csv, index=False)

print("Saved:", summary_csv)
print("Saved:", scenario_csv)

In [ ]:
import json
import pandas as pd
from pathlib import Path

MODEL_TO_VIEW = "yolov8n"
combined_path = Path(OUT_ROOT) / f"patch1_cross_scenario_transfer_{MODEL_TO_VIEW}" / "combined_metrics.json"
data = json.loads(combined_path.read_text())

rows = []
for s in data["per_scenario"]:
    if s["scenario"] == "billboard01":
        continue
    rows.append({
        "Scenario": s["scenario"],
        "mAP50_C": s["clean"]["yolo_val"]["mAP50"],
        "mAP50_P": s["patched"]["yolo_val"]["mAP50"],
        "Δ_mAP50": s["delta_patched_minus_clean"]["yolo_val"]["mAP50"],
        "mAP50-95_C": s["clean"]["yolo_val"]["mAP50-95"],
        "mAP50-95_P": s["patched"]["yolo_val"]["mAP50-95"],
        "Δ_mAP50-95": s["delta_patched_minus_clean"]["yolo_val"]["mAP50-95"],
        "Precision_C": s["clean"]["yolo_val"]["precision"],
        "Precision_P": s["patched"]["yolo_val"]["precision"],
        "Recall_C": s["clean"]["yolo_val"]["recall"],
        "Recall_P": s["patched"]["yolo_val"]["recall"],
        "Det_img_C": s["clean"]["A"]["detections_per_image_mean"],
        "Det_img_P": s["patched"]["A"]["detections_per_image_mean"],
        "Δ_Det_img": s["delta_patched_minus_clean"]["A"]["detections_per_image_mean"],
        "FP_img_C": s["clean"]["B"]["fp_per_image_mean_conf_min"],
        "FP_img_P": s["patched"]["B"]["fp_per_image_mean_conf_min"],
        "Δ_FP_img": s["delta_patched_minus_clean"]["B"]["fp_per_image_mean_conf_min"],
    })

scenario_df = pd.DataFrame(rows)
scenario_df

In [ ]:
# CELL 10 — Optional: zip the output folder for easy download
import shutil
from pathlib import Path

zip_base = str(Path(OUT_ROOT).parent / "unified_eval_ab_bundle")
archive_path = shutil.make_archive(zip_base, "zip", OUT_ROOT)
print("Created:", archive_path)